In [ ]:
import jax
import jax.numpy as jnp
from jax import lax
import matplotlib.pyplot as plt
import time
import os
import pandas as pd
import numpy as np

Nx_v = [11,21,51,101,201,501,1001,2001,5001]
Nx_v = [11,21,51,101,201,501,1001]
#Nx_v = [11,21,51,101,201,501]
n_iter = 1

# --- axon parameters
L = 1000    #in µm
d = 0.5       # diameter in µm


#Inject current 1ms pulse 
position=L / 2
t_start = 1.0
duration = 1.0
amplitude = 2

# --- solver setup
tsim = 10.0         # total simulation time [ms]
dt = 0.001          # time step [ms]


Vinit = -70.0



Cm=1.0
Ra=100.0
a = d / 2
a_cm = a * 1e-4
cm = 2.0 * np.pi * a_cm * Cm * 1e-6     # [F/cm]
ra = Ra / (np.pi * a_cm**2)   
D = (1.0 / (ra * cm)) / 1000.0  

t_start_inj = t_start 
t_stop_inj = t_start + duration 


jax.config.update("jax_enable_x64", False)  # float32


In [ ]:
# --------------------------
# Safe exponential
# --------------------------
def safe_exp(x):
    """
    Safe exponential wrapper that clips large inputs to avoid overflow.
    Uses float64 precision by default when provided inputs are float64.
    """
    x = jnp.asarray(x)
    # clip range chosen to avoid overflow on common hardware
    x = jnp.clip(x, a_min=-100.0, a_max=100.0)
    return jnp.exp(x)


# --------------------------
# Current injection
# --------------------------
def Iinj_uAcm2(t, t_start_inj, t_stop_inj, Nx, inj_uA_per_cm2, idx_inj):
    """
    Return injected current density array [µA/cm²] of length Nx.
    If t is within [t_start_inj, t_stop_inj] then the element at idx_inj
    receives inj_uA_per_cm2, otherwise all zeros.

    This is JAX-friendly (uses .at[...] to set a single index).
    """
    t = jnp.asarray(t)
    mask = (t >= t_start_inj) & (t <= t_stop_inj)
    arr = jnp.zeros((Nx,), dtype=jnp.result_type(inj_uA_per_cm2, jnp.float64))
    # Use scatter update guarded by mask
    arr = jax.lax.cond(
        mask,
        lambda a: a.at[idx_inj].set(inj_uA_per_cm2),
        lambda a: a,
        arr
    )
    return arr


# --------------------------
# vtrap
# --------------------------
def vtrap(x, y):
    """
    Stable vtrap from NEURON modfile.
    returns x / (exp(x/y) - 1) but uses series expansion when z ~ 0.
    Works elementwise for arrays.
    """
    x = jnp.asarray(x)
    y = jnp.asarray(y)
    z = x / y
    small = jnp.abs(z) < 1e-6
    # Avoid calling jnp.exp on huge inputs: clip inside safe_exp
    exp_term = safe_exp(z)
    denom = exp_term - 1.0
    safe_val = x / denom
    series = y * (1.0 - z / 2.0)
    return jnp.where(small, series, safe_val)


# --------------------------
# Ionic currents
# --------------------------
def Iion(V, Nx, m, n, h):
    """
    Return ionic current density [µA/cm²] for V (mV).
    Inputs V, m, n, h can be scalars or 1D arrays (length Nx).
    """
    V = jnp.asarray(V, dtype=jnp.float64)
    # Ensure V has shape (Nx,)
    if V.shape == ():
        V = jnp.full((Nx,), V, dtype=jnp.float64)

    m = jnp.asarray(m, dtype=jnp.float64)
    n = jnp.asarray(n, dtype=jnp.float64)
    h = jnp.asarray(h, dtype=jnp.float64)

    # constants
    gnabar = 0.12   # S/cm^2
    gkbar = 0.036   # S/cm^2
    gl = 0.0003     # S/cm^2
    el = -59.4      # mV
    ena = 45.0      # mV
    ek = -82.0      # mV

    gna = gnabar * (m ** 3) * h
    gk = gkbar * (n ** 4)

    ina = gna * (V - ena) * 1e3
    ik = gk * (V - ek) * 1e3
    il = gl * (V - el) * 1e3

    return ina + ik + il


# --------------------------
# Gate updates
# --------------------------
def update_gate_halfstep(g_prev, alpha_fun, beta_fun, V, dt):
    """
    Half-step gate update used by Crank-Nicholson scheme.
    g_prev, V can be arrays of same shape.
    """
    celsius = 37.0
    q10 = 2.24659524757 ** ((celsius - 6.3) / 10.0)
    alpha = q10 * alpha_fun(V)
    beta = q10 * beta_fun(V)

    # denom might be small, clamp to avoid division by zero
    denom = (1.0 / dt) + 0.5 * (alpha + beta)
    denom = jnp.maximum(denom, 1e-12)

    term1 = alpha / denom
    term2 = ((1.0 / dt) - 0.5 * (alpha + beta)) / denom * g_prev
    return term1 + term2


def half_step_gates(dt_ms, V_mV, Nx, m, h, n):
    """
    Advance gating variables m,h,n one half-step.
    Returns updated (m, h, n) as jnp arrays.
    """
    V = jnp.asarray(V_mV, dtype=jnp.float64)
    # broadcast if scalar
    if V.shape == ():
        V = jnp.full((Nx,), V, dtype=jnp.float64)

    m = jnp.asarray(m, dtype=jnp.float64)
    h = jnp.asarray(h, dtype=jnp.float64)
    n = jnp.asarray(n, dtype=jnp.float64)

    m_new = update_gate_halfstep(m, alpha_m, beta_m, V, dt_ms)
    h_new = update_gate_halfstep(h, alpha_h, beta_h, V, dt_ms)
    n_new = update_gate_halfstep(n, alpha_n, beta_n, V, dt_ms)

    return m_new, h_new, n_new


# --------------------------
# Alpha / Beta functions
# --------------------------
def alpha_m(V_m):
    V_m = jnp.asarray(V_m, dtype=jnp.float64)
    return vtrap(2.5 - 0.1 * (V_m + 70.0), 1.0)


def beta_m(V_m):
    V_m = jnp.asarray(V_m, dtype=jnp.float64)
    return 4.0 * safe_exp(-(V_m + 70.0) / 18.0)


def alpha_h(V_m):
    V_m = jnp.asarray(V_m, dtype=jnp.float64)
    return 0.07 * safe_exp(-(V_m + 70.0) / 20.0)


def beta_h(V_m):
    V_m = jnp.asarray(V_m, dtype=jnp.float64)
    return 1.0 / (safe_exp(3.0 - 0.1 * (V_m + 70.0)) + 1.0)


def alpha_n(V_m):
    V_m = jnp.asarray(V_m, dtype=jnp.float64)
    return 0.1 * vtrap(1.0 - 0.1 * (V_m + 70.0), 1.0)


def beta_n(V_m):
    V_m = jnp.asarray(V_m, dtype=jnp.float64)
    return 0.125 * safe_exp(-(V_m + 70.0) / 80.0)


# --------------------------
# Rates
# --------------------------
def rates(V_mV):
    """
    Compute (minf, mtau, hinf, htau, ninf, ntau) from V.
    V_mV can be scalar or array; returns arrays of same shape.
    """
    v = jnp.asarray(V_mV, dtype=jnp.float64)
    # broadcast scalars to 1D if necessary
    # (if you expect scalar V, you might want scalars back — keep arrays for consistency)
    celsius = 37.0
    q10 = 2.24659524757 ** ((celsius - 6.3) / 10.0)

    am = alpha_m(v)
    bm = beta_m(v)
    ah = alpha_h(v)
    bh = beta_h(v)
    an = alpha_n(v)
    bn = beta_n(v)

    sum_m = jnp.maximum(am + bm, 1e-12)
    sum_h = jnp.maximum(ah + bh, 1e-12)
    sum_n = jnp.maximum(an + bn, 1e-12)

    mtau = 1.0 / (q10 * sum_m)
    htau = 1.0 / (q10 * sum_h)
    ntau = 1.0 / (q10 * sum_n)

    minf = am / sum_m
    hinf = ah / sum_h
    ninf = an / sum_n

    return minf, mtau, hinf, htau, ninf, ntau


In [ ]:

def append_to_csv(df, filepath="benchmark.csv"):
    """
    Append or update a DataFrame in an existing CSV file, or create it if it doesn't exist.
    If the 'label' column in df already exists in the CSV, the old rows are replaced.

    Parameters
    ----------
    df : pandas.DataFrame
        The DataFrame to append or update. Must contain a 'label' column.
    filepath : str, optional
        Name of the CSV file (default: 'benchmark.csv').

    Returns
    -------
    pandas.DataFrame
        The combined DataFrame saved to CSV.
    """
    # Resolve the path relative to the script's own directory
    script_dir = os.path.dirname(os.path.abspath(__file__))
    full_path = os.path.join(script_dir, filepath)

    # If the file exists, load it
    if os.path.exists(full_path):
        existing_df = pd.read_csv(full_path)

        # Remove rows with the same labels as in the new df
        combined_df = pd.concat([
            existing_df[~existing_df['label'].isin(df['label'])],
            df
        ], ignore_index=True)
    else:
        combined_df = df.copy()

    # Write back to CSV
    combined_df.to_csv(full_path, index=False)

    return combined_df



def res_to_df(N_vec, T_vec, label) -> pd.DataFrame: 
    df = pd.DataFrame({
        "label": [label] * len(N_vec),
        "N": N_vec,
        "time": T_vec
    })
    
    return df

In [ ]:

# --------------------------
# Crank-Nicholson using tridiagonal_solve (optimized)
# --------------------------
def run_jax_tridiagonal_scan_optimized(Nx, Nt, alpha, inj_uA_per_cm2):
    x_axon = jnp.linspace(0.0, L, Nx, dtype=jnp.float32)
    idx_inj = jnp.argmin(jnp.abs(x_axon - position))

    # Tridiagonal vectors
    dl = -alpha * jnp.ones(Nx, dtype=jnp.float32).at[0].set(0.0)
    d  = (1 + 2*alpha) * jnp.ones(Nx, dtype=jnp.float32)
    du = -alpha * jnp.ones(Nx, dtype=jnp.float32).at[-1].set(0.0)

    # Voltage and gating variables
    V = jnp.full((Nx,), Vinit, dtype=jnp.float32)
    minf, mtau, hinf, htau, ninf, ntau = rates(V)
    m_RA, h_RA, n_RA = minf, hinf, ninf

    # Preallocate output
    V_all = jnp.zeros((Nt, Nx), dtype=jnp.float32)

    # --------------------------
    def step(carry, n):
        V, m_RA, h_RA, n_RA = carry
        t_mid = n * dt

        # Currents
        Iinj = Iinj_uAcm2(t_mid, t_start_inj, t_stop_inj, Nx, inj_uA_per_cm2, idx_inj)
        Iion_curr = Iion(V, Nx, m_RA, n_RA, h_RA)

        rhs = V + (dt / (2.0*Cm)) * (Iinj - Iion_curr)

        # Update gating variables
        m_RA, h_RA, n_RA = half_step_gates(dt, V, Nx, m_RA, h_RA, n_RA)

        # Apply boundary conditions
        rhs = rhs.at[0].set(Vinit).at[-1].set(Vinit)

        # Solve tridiagonal system
        V_half = jax.lax.linalg.tridiagonal_solve(dl, d, du, rhs[:, None])[:, 0]

        # Crank-Nicholson update
        V_new = 2.0 * V_half - V
        V_new = jnp.clip(V_new, -500.0, 500.0)
        V_new = V_new.at[0].set(Vinit).at[-1].set(Vinit)

        return (V_new, m_RA, h_RA, n_RA), V_new  # carry without V_all, output is V_new

    # --------------------------
    # Scan over time
    (V_final, m_final, h_final, n_final), V_all = lax.scan(step, (V, m_RA, h_RA, n_RA), jnp.arange(Nt))
    t_vec = jnp.arange(Nt, dtype=jnp.float32) * dt
    return t_vec, V_all  # V_all already stacked

# --------------------------
# Benchmark
# --------------------------
t_v = []
res_list = []

for Nx in Nx_v:
    dx = L / (Nx - 1)
    dx_cm = dx * 1e-4
    dx2 = dx_cm**2
    alpha = D * (dt / 2.0) / dx2
    inj_uA_per_cm2 = amplitude * 1e-3 / (2.0 * jnp.pi * a_cm * dx_cm)
    Nt = int(jnp.ceil(tsim / dt))

    run_fn_jit = jax.jit(lambda Nx=Nx, Nt=Nt, alpha=alpha, inj_uA_per_cm2=inj_uA_per_cm2:
                         run_jax_tridiagonal_scan_optimized(Nx, Nt, alpha, inj_uA_per_cm2))

    # Compile first
    res_compiled = run_fn_jit()
    res_compiled[1].block_until_ready()
    start_time = time.perf_counter()
    res = run_fn_jit()
    res[1].block_until_ready()
    end_time = time.perf_counter()

    print(f"Nx={Nx}: Execution time (tridiagonal float32 ultra-optimized) = {end_time - start_time:.4f} s")
    t_v.append(end_time - start_time)
    res_list.append(res)

# --------------------------
# Save benchmark results
# --------------------------
#df = u.res_to_df(Nx_v, t_v, label="jax_tridiagonal_jit_ultra_f32")
#u.append_to_csv(df)

# --------------------------
# Plot example
# --------------------------
t_vec, V_all = res_list[-1]
x_axon = jnp.linspace(0.0, L, Nx_v[-1], dtype=jnp.float32)
x_positions = [L/4, L/3, L/2, 2*L/3, 3*L/4]
indices = [jnp.argmin(jnp.abs(x_axon - xp)) for xp in x_positions]

fig, ax = plt.subplots(1, figsize=(5,5))
for idx, xp in zip(indices, x_positions):
    ax.plot(t_vec, V_all[:, idx], label=f'x={xp:.3f} cm')
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Voltage (mV)")
ax.legend()
